In [1]:
import time
import os
import requests
import pandas as pd
import csv

Lấy dữ liệu IMDB

In [7]:
basics_path = 'title.basics.tsv.gz'
ratings_path = 'title.ratings.tsv.gz'
output_path = 'imdb_raw.csv'
# DATA_DIR = '../data/raw'
# basics_path = os.path.join(DATA_DIR, 'title.basics.tsv.gz')
# ratings_path = os.path.join(DATA_DIR, 'title.ratings.tsv.gz')
# output_path = os.path.join(DATA_DIR, 'imdb_raw.csv')

df_basics = pd.read_csv(
    basics_path,
    sep='\t',
    compression='gzip',
    na_values='\\N',
    usecols=['tconst', 'titleType', 'startYear', 'runtimeMinutes'],
    dtype={
        'tconst': 'string',
        'titleType': 'category',
        'startYear': 'Int32',
        'runtimeMinutes': 'Float32'
    },
    quoting=csv.QUOTE_NONE
)

df_basics = df_basics[
    (df_basics['titleType'] == 'movie') &
    (df_basics['startYear'] >= 2015) &
    (df_basics['startYear'] <= 2025)
].copy()
df_basics = df_basics.drop(columns=['titleType', 'startYear'])

df_ratings = pd.read_csv(
    ratings_path,
    sep='\t',
    compression='gzip',
    na_values='\\N',
    usecols=['tconst', 'numVotes'],
    dtype={
        'tconst': 'string',
        'numVotes': 'Int32'
    }
)

df_ratings = df_ratings[df_ratings['numVotes'] >= 5000]
df_merged = pd.merge(df_basics, df_ratings, on='tconst', how='inner')
df_merged = df_merged.drop(columns=['numVotes'])

del df_basics, df_ratings

print(f"-> Tổng số phim: {len(df_merged)}")

df_merged.to_csv(output_path, index=False)
df_merged.head()

-> Tổng số phim: 6182


,tconst,runtimeMinutes
0,tt0069049,122.0
1,tt0293429,110.0
2,tt0315642,103.0
3,tt0327785,104.0
4,tt0360556,100.0


Lấy dữ liệu từ API TMDB

In [12]:
API_KEY = '9dcc9b3b89d9f66fa1d8a2bf4c11b407'

INPUT_FILE = 'imdb_raw.csv'
OUTPUT_FILE = 'tmdb_raw.csv'
CHECKPOINT_FILE = 'tmdb_checkpoint.csv'
# DATA_DIR = '../data/raw'
# INPUT_FILE = os.path.join(DATA_DIR, 'imdb_raw.csv')
# OUTPUT_FILE = os.path.join(DATA_DIR, 'tmdb_raw.csv')
# CHECKPOINT_FILE = os.path.join(DATA_DIR, 'tmdb_checkpoint.csv')

# ==========================================
# BẢNG ÁNH XẠ NGÔN NGỮ
# ==========================================
LANG_TO_COUNTRY_MAP = {
    'en': ['US', 'GB', 'AU', 'CA', 'NZ', 'IE'],
    'zh': ['CN', 'HK', 'TW', 'SG'], 'cn': ['CN', 'HK', 'TW', 'SG'], 'yue': ['HK', 'CN'],
    'hi': ['IN'], 'ta': ['IN'], 'te': ['IN'], 'ml': ['IN'], 'kn': ['IN'],
    'bn': ['IN', 'BD'], 'pa': ['IN'], 'ur': ['IN', 'PK'],
    'fr': ['FR', 'BE', 'CH', 'CA'],
    'es': ['ES', 'MX', 'AR', 'CO', 'CL', 'PE'],
    'de': ['DE', 'AT', 'CH'],
    'pt': ['BR', 'PT'],
    'it': ['IT'],
    'ru': ['RU', 'BY', 'KZ'],
    'nl': ['NL', 'BE'],
    'sv': ['SE'], 'da': ['DK'], 'no': ['NO'], 'fi': ['FI'],
    'pl': ['PL'], 'tr': ['TR'], 'cs': ['CZ'], 'hu': ['HU'], 'el': ['GR'],
    'ja': ['JP'], 'ko': ['KR'], 'th': ['TH'], 'vi': ['VN'], 'id': ['ID'],
    'tl': ['PH'], 'ms': ['MY', 'SG'],
    'ar': ['EG', 'SA', 'AE', 'LB', 'MA'], 'fa': ['IR'], 'he': ['IL']
}

def get_movie_data(imdb_id):
    start_total = time.time()

    try:
        t_find_start = time.time()
        r = requests.get(
            f"https://api.themoviedb.org/3/find/{imdb_id}",
            params={"api_key": API_KEY, "external_source": "imdb_id"},
            timeout=15
        )
        t_find = time.time() - t_find_start

        if r.status_code == 429:
            print(f"  [!] Rate Limit ở Find API, chờ retry...")
            time.sleep(int(r.headers.get("Retry-After", 10)))
            return get_movie_data(imdb_id)

        movie_results = r.json().get('movie_results', [])
        if not movie_results:
            return None
        tmdb_id = movie_results[0]['id']
    except Exception as e:
        print(f"  find error {imdb_id}: {e}")
        return None

    try:
        t_detail_start = time.time()
        r2 = requests.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={"api_key": API_KEY, "append_to_response": "release_dates,credits,keywords"},
            timeout=15
        )
        t_detail = time.time() - t_detail_start

        if r2.status_code == 429:
            print(f"  [!] Rate Limit ở Detail API, chờ retry...")
            time.sleep(int(r2.headers.get("Retry-After", 10)))
            r2 = requests.get(
                f"https://api.themoviedb.org/3/movie/{tmdb_id}",
                params={"api_key": API_KEY, "append_to_response": "release_dates,credits,keywords"},
                timeout=15
            )
        res = r2.json()
    except Exception as e:
        print(f"  detail error {imdb_id}: {e}")
        return None

    origin_countries = res.get('origin_country', [])
    original_lang = res.get('original_language', '')
    companies = [c['id'] for c in res.get('production_companies', [])] or None
    cast_raw = res.get('credits', {}).get('cast', [])[:3]
    top3_cast = [{'id': m['id'], 'popularity': m['popularity']} for m in cast_raw] or None
    crew_raw = res.get('credits', {}).get('crew', [])
    directors = [{'id': m['id'], 'popularity': m['popularity']}
                 for m in crew_raw if m.get('job') == 'Director'] or None
    keywords_raw = res.get('keywords', {}).get('keywords', [])
    keywords = [k['id'] for k in keywords_raw] or None
    genres = [g['id'] for g in res.get('genres', [])] or None
    release_results = res.get('release_dates', {}).get('results', [])

    certification = None
    for entry in release_results:
        if entry['iso_3166_1'] == 'US':
            for d in entry['release_dates']:
                cert = d.get('certification', '').strip()
                if cert:
                    certification = cert
                    break
            break

    # LOGIC TÌM THEATRICAL RELEASE (TYPE 3)
    type3_dates = {}
    for entry in release_results:
        country_code = entry['iso_3166_1']
        for d in entry['release_dates']:
            if d.get('type') == 3:
                type3_dates[country_code] = d.get('release_date', '').split('T')[0]
                break

    target_countries = LANG_TO_COUNTRY_MAP.get(original_lang, [])
    num_origin = len(origin_countries)
    theatrical_release = None

    # US + en
    if original_lang == 'en' and 'US' in origin_countries and 'US' in type3_dates:
        theatrical_release = type3_dates['US']

    # 1 nước
    elif num_origin == 1:
        c = origin_countries[0]
        if c in type3_dates:
            theatrical_release = type3_dates[c]

    # >= 2 nước (Ưu tiên khớp ngôn ngữ, lấy nước đứng trước)
    if not theatrical_release and num_origin > 1:
        matches = [c for c in origin_countries if c in target_countries]
        for m in matches:
            if m in type3_dates:
                theatrical_release = type3_dates[m]
                break

    # List doanh thu cao
    if not theatrical_release and num_origin > 1:
        priority_list = ['US', 'CN', 'JP', 'IN', 'GB', 'FR', 'DE', 'MX', 'KR', 'AU']
        for p_country in priority_list:
            if p_country in origin_countries and p_country in type3_dates:
                theatrical_release = type3_dates[p_country]
                break

    # Fallback
    if not theatrical_release:
        for c in origin_countries:
            if c in type3_dates:
                theatrical_release = type3_dates[c]
                break
        if not theatrical_release and type3_dates:
            theatrical_release = list(type3_dates.values())[0]

    return {
        'budget':               res.get('budget') or None,
        'revenue':              res.get('revenue') or None,
        'production_companies': companies,
        'original_language':    original_lang,
        'origin_country':       origin_countries,
        'top_3_cast':           top3_cast,
        'director':             directors,
        'keywords':             keywords,
        'genres':               genres,
        'certification_us':     certification,
        'theatrical_release':   theatrical_release,
    }

df = pd.read_csv(INPUT_FILE)

start_index = 0
results = []
# if os.path.exists(CHECKPOINT_FILE):
#     checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
#     start_index = len(checkpoint_df)
#     results = checkpoint_df.to_dict('records')
#     print(f"Resume từ dòng {start_index}/{len(df)}")

print(f"Lấy dữ liệu {len(df) - start_index} phim...")

start_time = time.time()
batch_start_time = time.time()

for i in range(start_index, len(df)):
    iter_start = time.time()
    imdb_id = df.iloc[i]['tconst']
    stats = get_movie_data(imdb_id)

    entry = {'tconst': imdb_id}
    if stats:
        entry.update(stats)
    results.append(entry)

    time.sleep(0.04)

    if (i + 1) % 100 == 0:
        current_speed = 100 / (time.time() - batch_start_time)
        print(f"{i+1}/{len(df) - start_index} - Tốc độ: {current_speed:.2f} dòng/s")
        batch_start_time = time.time()

    if (i + 1) % 500 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"--- ĐÃ LƯU CHECKPOINT TẠI DÒNG {i+1} ---")

total_time = (time.time() - start_time)
print(f"Thời gian chạy: {int(total_time/60)}m {int(total_time%60)}s")
final_df = pd.DataFrame(results)
final_df.to_csv(OUTPUT_FILE, index=False)
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
print(f"Xong! Đã lưu {len(final_df)} dòng vào '{OUTPUT_FILE}'")

Lấy dữ liệu 6182 phim...
100/6182 - Tốc độ: 11.48 dòng/s
200/6182 - Tốc độ: 11.51 dòng/s
300/6182 - Tốc độ: 10.57 dòng/s
400/6182 - Tốc độ: 9.97 dòng/s
500/6182 - Tốc độ: 11.33 dòng/s
--- ĐÃ LƯU CHECKPOINT TẠI DÒNG 500 ---
600/6182 - Tốc độ: 11.53 dòng/s
700/6182 - Tốc độ: 11.16 dòng/s
800/6182 - Tốc độ: 11.66 dòng/s
900/6182 - Tốc độ: 10.54 dòng/s
1000/6182 - Tốc độ: 9.59 dòng/s
--- ĐÃ LƯU CHECKPOINT TẠI DÒNG 1000 ---
1100/6182 - Tốc độ: 9.52 dòng/s
1200/6182 - Tốc độ: 9.51 dòng/s
1300/6182 - Tốc độ: 9.58 dòng/s
1400/6182 - Tốc độ: 9.44 dòng/s
1500/6182 - Tốc độ: 9.42 dòng/s
--- ĐÃ LƯU CHECKPOINT TẠI DÒNG 1500 ---
1600/6182 - Tốc độ: 9.56 dòng/s
1700/6182 - Tốc độ: 9.47 dòng/s
1800/6182 - Tốc độ: 9.41 dòng/s
1900/6182 - Tốc độ: 9.47 dòng/s
2000/6182 - Tốc độ: 9.34 dòng/s
--- ĐÃ LƯU CHECKPOINT TẠI DÒNG 2000 ---
2100/6182 - Tốc độ: 9.50 dòng/s
2200/6182 - Tốc độ: 9.65 dòng/s
2300/6182 - Tốc độ: 9.52 dòng/s
2400/6182 - Tốc độ: 9.52 dòng/s
2500/6182 - Tốc độ: 9.46 dòng/s
--- ĐÃ LƯU CHECKP

Merge

In [13]:
imdb_df = pd.read_csv('imdb_raw.csv')
tmdb_df = pd.read_csv('tmdb_raw.csv')

movie_raw = imdb_df.merge(tmdb_df, on='tconst', how='left')
movie_raw.to_csv('movie_raw.csv', index=False)
print(f"Xong! Lưu {len(movie_raw)} dòng vào 'movie_raw.csv'")

Xong! Lưu 6182 dòng vào 'movie_raw.csv'
